In [84]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import math

In [94]:
np.random.seed(42)

days = 365
beds = 75
max_days_padded = days + 100


def generate_arrivals(t, ward):
    if ward == 'A':
        lambda_A = -(1/3650) * t**2 + (1/10) * t
        return np.random.poisson(max(0, lambda_A))
    if ward == 'B':
        lambda_A = -(1/3650) * t**2 + (1/10) * t
        lambda_B = (1/5) * lambda_A
        return np.random.poisson(max(0, lambda_B))
    if ward == 'C':
        return np.random.poisson(6)
        
def LOS(ward, arrivals):
    if arrivals == 0:
        return np.array([], dtype=int)
    
    sigma = np.sqrt(np.log(2))
    if ward == 'A':
        mu = np.log(4 * np.sqrt(2))
    elif ward == 'B':
        mu = np.log(6 * np.sqrt(2))
    elif ward == 'C':
        mu = np.log(5 * np.sqrt(2))
        
    los_days = np.ceil(np.random.lognormal(mean=mu, sigma=sigma, size=arrivals)).astype(int)
    return np.maximum(1, los_days)



def simulate_hospital(generate_arrivals, LOS, beds_A_alloc, beds_B_alloc, beds_C_alloc):
    blocking_fraction = []
    beds_A = np.full(max_days_padded, beds_A_alloc)
    beds_B = np.full(max_days_padded, beds_B_alloc)
    beds_C = np.full(max_days_padded, beds_C_alloc)

    total_arrivals = {'A': 0, 'B': 0, 'C': 0}
    blocked_counts = {'A': 0, 'B': 0, 'C': 0}
    
    utilization_A = []
    utilization_B = []
    utilization_C = []

    for t in range(0,days):
        blocked = 0
        arrivals_A = generate_arrivals(t,'A')
        arrivals_B = generate_arrivals(t,'B')
        arrivals_C = generate_arrivals(t,'C')

        total_arrivals['A'] += arrivals_A
        total_arrivals['B'] += arrivals_B
        total_arrivals['C'] += arrivals_C

        los_A = LOS('A', arrivals_A)
        los_B = LOS('B', arrivals_B)
        los_C = LOS('C', arrivals_C)

        for length in los_A:
            if np.all(beds_A[t:t+length] > 0):
                beds_A[t:t+length] -= 1
            else:
                blocked_counts['A'] += 1


        for length in los_B:
            if np.all(beds_B[t:t+length] > 0):
                beds_B[t:t+length] -= 1
            else:
                if np.all(beds_A[t:t+length] > 0):
                    beds_A[t:t+length] -= 1
                else:
                    blocked_counts['B'] += 1

        for length in los_C:
            if np.all(beds_C[t:t+length] > 0):
                beds_C[t:t+length] -= 1
            else:
                blocked_counts['C'] += 1
        
        utilization_A.append((beds_A_alloc - beds_A[t]) / beds_A_alloc)
        utilization_B.append((beds_B_alloc - beds_B[t]) / beds_B_alloc)
        utilization_C.append((beds_C_alloc - beds_C[t]) / beds_C_alloc)
                
    total_blocked = sum(blocked_counts.values())
    blocked_frac = total_blocked/sum(total_arrivals.values())
    blocked_ward_A = blocked_counts['A']/total_arrivals['A']
    blocked_ward_B = blocked_counts['B']/total_arrivals['B']
    blocked_ward_C = blocked_counts['C']/total_arrivals['C']
    mean_util_A = np.mean(utilization_A)
    mean_util_B = np.mean(utilization_B)
    mean_util_C = np.mean(utilization_C)

    return total_blocked, blocked_frac, blocked_ward_A, blocked_ward_B, blocked_ward_C, mean_util_A, mean_util_B, mean_util_C


In [102]:
import numpy as np

np.random.seed(42)

days = 365
beds = 75
max_days_padded = days + 100


def generate_arrivals(t, ward):
    if ward == 'A':
        lambda_A = -(1/3650) * t**2 + (1/10) * t
        return np.random.poisson(max(0, lambda_A))
    if ward == 'B':
        lambda_A = -(1/3650) * t**2 + (1/10) * t
        lambda_B = (1/5) * lambda_A
        return np.random.poisson(max(0, lambda_B))
    if ward == 'C':
        return np.random.poisson(6)
        
def LOS(ward, arrivals):
    if arrivals == 0:
        return np.array([], dtype=int)
    
    sigma = np.sqrt(np.log(2))
    if ward == 'A':
        mu = np.log(4 * np.sqrt(2))
    elif ward == 'B':
        mu = np.log(6 * np.sqrt(2))
    elif ward == 'C':
        mu = np.log(5 * np.sqrt(2))
        
    los_days = np.ceil(np.random.lognormal(mean=mu, sigma=sigma, size=arrivals)).astype(int)
    return np.maximum(1, los_days)



def simulate_hospital(generate_arrivals, LOS, beds_A_alloc, beds_B_alloc, beds_C_alloc):
    blocking_fraction = []
    beds_A = np.full(max_days_padded, beds_A_alloc)
    beds_B = np.full(max_days_padded, beds_B_alloc)
    beds_C = np.full(max_days_padded, beds_C_alloc)

    total_arrivals = {'A': 0, 'B': 0, 'C': 0}
    blocked_counts = {'A': 0, 'B': 0, 'C': 0}
    
    utilization_A = []
    utilization_B = []
    utilization_C = []

    for t in range(0,days):
        blocked = 0
        arrivals_A = generate_arrivals(t,'A')
        arrivals_B = generate_arrivals(t,'B')
        arrivals_C = generate_arrivals(t,'C')

        total_arrivals['A'] += arrivals_A
        total_arrivals['B'] += arrivals_B
        total_arrivals['C'] += arrivals_C

        los_A = LOS('A', arrivals_A)
        los_B = LOS('B', arrivals_B)
        los_C = LOS('C', arrivals_C)

        for length in los_A:
            # CHANGE: Check if bed is available ONLY at arrival time 't', not 'np.all' across the stay
            if beds_A[t] > 0:
                beds_A[t:t+length] -= 1
            else:
                blocked_counts['A'] += 1


        for length in los_B:
            # CHANGE: Check Ward B capacity ONLY at arrival time 't'
            if beds_B[t] > 0:
                beds_B[t:t+length] -= 1
            else:
                # CHANGE: Check Ward A overflow capacity ONLY at arrival time 't'
                if beds_A[t] > 0:
                    beds_A[t:t+length] -= 1
                else:
                    blocked_counts['B'] += 1

        for length in los_C:
            # CHANGE: Check Ward C capacity ONLY at arrival time 't'
            if beds_C[t] > 0:
                beds_C[t:t+length] -= 1
            else:
                blocked_counts['C'] += 1
        
        utilization_A.append((beds_A_alloc - beds_A[t]) / beds_A_alloc)
        utilization_B.append((beds_B_alloc - beds_B[t]) / beds_B_alloc)
        utilization_C.append((beds_C_alloc - beds_C[t]) / beds_C_alloc)
                
    total_blocked = sum(blocked_counts.values())
    
    # CHANGE: Added safety checks (if > 0) to prevent potential ZeroDivisionError 
    blocked_frac = total_blocked/sum(total_arrivals.values()) if sum(total_arrivals.values()) > 0 else 0
    blocked_ward_A = blocked_counts['A']/total_arrivals['A'] if total_arrivals['A'] > 0 else 0
    blocked_ward_B = blocked_counts['B']/total_arrivals['B'] if total_arrivals['B'] > 0 else 0
    blocked_ward_C = blocked_counts['C']/total_arrivals['C'] if total_arrivals['C'] > 0 else 0
    
    mean_util_A = np.mean(utilization_A)
    mean_util_B = np.mean(utilization_B)
    mean_util_C = np.mean(utilization_C)

    return total_blocked, blocked_frac, blocked_ward_A, blocked_ward_B, blocked_ward_C, mean_util_A, mean_util_B, mean_util_C

In [101]:
b_A = 35
b_B = 2
b_C = 38
simulate_hospital(generate_arrivals, LOS, b_A, b_B, b_C)

(2174,
 0.44704914661731443,
 0.43873873873873875,
 0.7203579418344519,
 0.3998178506375228,
 0.8706066536203523,
 0.9,
 0.9700793078586878)

In [103]:
best_config = None
min_relocated = float('inf')
results_log = []

step = 1
n_reps = 5  



for b_A in range(1, 75, step):
    for b_B in range(1, 75 - b_A, step):
        b_C = 75 - (b_A + b_B)
        
        if b_C < 5: 
            continue
            
        rep_blocked = []
        sim_frac = []
        ward_A = []
        ward_B = []
        ward_C = []
        util_A = []
        util_B = []
        util_C = []
        for rep in range(n_reps):
            sim_blocked, sim_blocked_frac, sim_ward_A, sim_ward_B, sim_ward_C, sim_util_A, sim_util_B, sim_util_C = simulate_hospital(generate_arrivals, LOS, b_A, b_B, b_C)
            rep_blocked.append(sim_blocked)
            sim_frac.append(sim_blocked_frac)
            ward_A.append(sim_ward_A)
            ward_B.append(sim_ward_B)
            ward_C.append(sim_ward_C)
            util_A.append(sim_util_A)
            util_B.append(sim_util_B)
            util_C.append(sim_util_C)
            
        avg_blocked = np.mean(rep_blocked)
        avg_frac = np.mean(sim_frac)
        avg_A = np.mean(ward_A)
        avg_B = np.mean(ward_B)
        avg_C = np.mean(ward_C)
        avg_util_A = np.mean(util_A)
        avg_util_B = np.mean(util_B)
        avg_util_C = np.mean(util_C)

        results_log.append(((b_A, b_B, b_C), avg_blocked, avg_frac, ward_A, ward_B, ward_C))
        
        if avg_blocked < min_relocated:
            min_relocated = avg_blocked
            best_config = (b_A, b_B, b_C)



In [104]:
print("\n--- Optimization Complete ---")
print(f"Optimal Bed Distribution (Ward A, Ward B, Ward C): {best_config}")
print(f"Minimum expected relocated patients: {min_relocated:.2f}")
print(f"Blocked fraction total: {avg_frac:.2f}")
print(f"Blocked frac from ward A: {avg_A:.2f}")
print(f"Blocked frac from ward B: {avg_B:.2f}")
print(f"Blocked frac from ward C: {avg_C:.2f}")
print(f"Utilization ward A: {avg_util_A:.2f}")
print(f"Utilization ward B: {avg_util_B:.2f}")
print(f"Utilization ward C: {avg_util_C:.2f}")



--- Optimization Complete ---
Optimal Bed Distribution (Ward A, Ward B, Ward C): (30, 2, 43)
Minimum expected relocated patients: 2111.00
Blocked fraction total: 0.50
Blocked frac from ward A: 0.12
Blocked frac from ward B: 0.42
Blocked frac from ward C: 0.91
Utilization ward A: 0.79
Utilization ward B: 0.93
Utilization ward C: 1.00


# Exponential

In [88]:
        
def LOS_exp(ward, arrivals):
    if arrivals == 0:
        return np.array([], dtype=int)
    
    sigma = np.sqrt(np.log(2))
    if ward == 'A':
        mu = 8
    elif ward == 'B':
        mu = 12
    elif ward == 'C':
        mu = 10
    
    los_days = np.ceil(np.random.exponential(scale=mu, size=arrivals)).astype(int)
    return np.maximum(1, los_days)




In [98]:
best_config = None
min_relocated = float('inf')
results_log = []

step = 1
n_reps = 5  



for b_A in range(1, 75, step):
    for b_B in range(1, 75 - b_A, step):
        b_C = 75 - (b_A + b_B)
        
        if b_C < 5: 
            continue
            
        rep_blocked = []
        sim_frac = []
        ward_A = []
        ward_B = []
        ward_C = []
        util_A = []
        util_B = []
        util_C = []
        for rep in range(n_reps):
            sim_blocked, sim_blocked_frac, sim_ward_A, sim_ward_B, sim_ward_C, sim_util_A, sim_util_B, sim_util_C = simulate_hospital(generate_arrivals, LOS_exp, b_A, b_B, b_C)
            rep_blocked.append(sim_blocked)
            sim_frac.append(sim_blocked_frac)
            ward_A.append(sim_ward_A)
            ward_B.append(sim_ward_B)
            ward_C.append(sim_ward_C)
            util_A.append(sim_util_A)
            util_B.append(sim_util_B)
            util_C.append(sim_util_C)
            
        avg_blocked = np.mean(rep_blocked)
        avg_frac = np.mean(sim_frac)
        avg_A = np.mean(ward_A)
        avg_B = np.mean(ward_B)
        avg_C = np.mean(ward_C)
        avg_util_A = np.mean(util_A)
        avg_util_B = np.mean(util_B)
        avg_util_C = np.mean(util_C)

        results_log.append(((b_A, b_B, b_C), avg_blocked, avg_frac, ward_A, ward_B, ward_C))
        
        if avg_blocked < min_relocated:
            min_relocated = avg_blocked
            best_config = (b_A, b_B, b_C)



print("\n--- Optimization Complete ---")
print(f"Optimal Bed Distribution (Ward A, Ward B, Ward C): {best_config}")
print(f"Minimum expected relocated patients: {min_relocated:.2f}")
print(f"Blocked fraction total: {avg_frac:.2f}")
print(f"Blocked frac from ward A: {avg_A:.2f}")
print(f"Blocked frac from ward B: {avg_B:.2f}")
print(f"Blocked frac from ward C: {avg_C:.2f}")
print(f"Utilization ward A: {avg_util_A:.2f}")
print(f"Utilization ward B: {avg_util_B:.2f}")
print(f"Utilization ward C: {avg_util_C:.2f}")



--- Optimization Complete ---
Optimal Bed Distribution (Ward A, Ward B, Ward C): (27, 4, 44)
Minimum expected relocated patients: 2129.80
Blocked fraction total: 0.52
Blocked frac from ward A: 0.13
Blocked frac from ward B: 0.40
Blocked frac from ward C: 0.92
Utilization ward A: 0.77
Utilization ward B: 0.94
Utilization ward C: 1.00


# More beds

In [ ]:
np.random.seed(42)

# Ændre her ift antal af senge
more_beds = 100

best_config = None
min_relocated = float('inf')
results_log = []

step = 1
n_reps = 5  



for b_A in range(1, more_beds, step):
    for b_B in range(1, more_beds - b_A, step):
        b_C = more_beds - (b_A + b_B)
        
        if b_C < 5: 
            continue
            
        rep_blocked = []
        sim_frac = []
        ward_A = []
        ward_B = []
        ward_C = []
        util_A = []
        util_B = []
        util_C = []
        for rep in range(n_reps):
            sim_blocked, sim_blocked_frac, sim_ward_A, sim_ward_B, sim_ward_C, sim_util_A, sim_util_B, sim_util_C = simulate_hospital(generate_arrivals, LOS, b_A, b_B, b_C)
            rep_blocked.append(sim_blocked)
            sim_frac.append(sim_blocked_frac)
            ward_A.append(sim_ward_A)
            ward_B.append(sim_ward_B)
            ward_C.append(sim_ward_C)
            util_A.append(sim_util_A)
            util_B.append(sim_util_B)
            util_C.append(sim_util_C)
            
        avg_blocked = np.mean(rep_blocked)
        avg_frac = np.mean(sim_frac)
        avg_A = np.mean(ward_A)
        avg_B = np.mean(ward_B)
        avg_C = np.mean(ward_C)
        avg_util_A = np.mean(util_A)
        avg_util_B = np.mean(util_B)
        avg_util_C = np.mean(util_C)

        results_log.append(((b_A, b_B, b_C), avg_blocked, avg_frac, ward_A, ward_B, ward_C))
        
        if avg_blocked < min_relocated:
            min_relocated = avg_blocked
            best_config = (b_A, b_B, b_C)



print("\n--- Optimization Complete ---")
print(f"Optimal Bed Distribution (Ward A, Ward B, Ward C): {best_config}")
print(f"Minimum expected relocated patients: {min_relocated:.2f}")
print(f"Blocked fraction total: {avg_frac:.2f}")
print(f"Blocked frac from ward A: {avg_A:.2f}")
print(f"Blocked frac from ward B: {avg_B:.2f}")
print(f"Blocked frac from ward C: {avg_C:.2f}")
print(f"Utilization ward A: {avg_util_A:.2f}")
print(f"Utilization ward B: {avg_util_B:.2f}")
print(f"Utilization ward C: {avg_util_C:.2f}")



--- Optimization Complete ---
Optimal Bed Distribution (Ward A, Ward B, Ward C): (30, 2, 43)
Minimum expected relocated patients: 2111.00
Blocked fraction total: 0.50
Blocked frac from ward A: 0.12
Blocked frac from ward B: 0.42
Blocked frac from ward C: 0.91
Utilization ward A: 0.79
Utilization ward B: 0.93
Utilization ward C: 1.00
